### Package Installation (commented out)  
- **openai** — the official Python client for calling OpenAI's GPT models via API
- **google-genai** and **jsonschema** — Google's Generative AI client and a JSON schema validator (available for optional use)
- **pydantic** — a data validation library for enforcing structured data types

Once the environment is already set up, these lines stay commented out to avoid re-installing on every run.

In [ ]:
# %pip install -q openai
# %pip install -q google-genai jsonschema
# %pip install -q pydantic

In [31]:
import os, getpass, json, re
import time
import textwrap
from pathlib import Path
from typing import Dict, List, Tuple

# for saving the API key in the .env file
from dotenv import load_dotenv

from openai import OpenAI



### Model  
- **MODEL** is set to `"gpt-4o"`, meaning all advertisement generation calls will use OpenAI's GPT-4o model. note that `gpt-4.1-nano` is the cheapest alternative if cost is a concern.
- **OUT_DIR** defines the folder (`stimuli_out`) where all generated advertisement JSON files will be saved.
- The cell then loads the OpenAI API key from the `.env` file (using `load_dotenv()`) and reads it into a variable — this is the credential that authenticates every API call. The Gemini key loading lines are also present but commented out, the notebook was designed to also support Google's model as well.

In [89]:
MODEL = "gpt-4o"   # or "gpt-4o", gpt-4.1-nano is cheapest
OUT_DIR = Path("stimuli_out")

# os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
# os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")

load_dotenv()
# Get the API key from the environment variable
api_key = os.getenv("OPENAI_API_KEY")

### Model Parameters & output parameters  
This cell defines three key parameters that govern how advertisements are generated and how many are produced:
- **TEMPERATURE = 0.8** controls the creativity/randomness of the GPT model's output. A value of 0.8 produces varied and natural-sounding text while remaining coherent — lower values produce more predictable output, higher values more creative/unpredictable.
- **WORD_RANGE = (90, 110)** sets the strict word count window that every generated advertisement must fall within. Any output outside 90–110 words will be rejected and re-generated.
- **PARAPHRASE_IDS = [1, 2, 3]** defines how many distinct paraphrase versions will be generated for each product/variant combination. Setting it to `[1]` (as the comment suggests) lets you test with a small batch first before generating all three.

In [138]:
TEMPERATURE = 0.7
WORD_RANGE = (90, 110)
PARAPHRASE_IDS = [1, 2, 3]  # set [1] to start small

## Define PRODUCTS FACTS  
This cell defines the catalogue of six products that advertisements will be generated for. Each product entry contains two pieces of information:
- **product_name** — the human-readable name of the product (e.g., "Fitness app", "Subscription coffee")
- **facts_line** — a single factual sentence describing the product's concrete features with no marketing spin (e.g., battery life, delivery schedule, cancellation policy)

The facts line is critical to the experiment's design: it must be reproduced **verbatim** in every generated advertisement regardless of personality targeting variant. This ensures that all ad versions for a given product contain identical factual information, so any differences between variants come purely from tone and framing — not from differing claims about the product.

In [34]:
PRODUCTS = {
  "P1": {"product_name": "Weekend city trip",
         "facts_line": "Two nights in a central hotel with breakfast, flexible dates, and free cancellation up to 48 hours before check-in."},
  "P2": {"product_name": "Fitness app",
         "facts_line": "Personalized workouts from 10–30 minutes, progress tracking, and adjustable plans for strength, cardio, or mobility."},
  "P3": {"product_name": "Noise-cancelling headphones",
         "facts_line": "Active noise cancellation, transparency mode, 30-hour battery, and comfortable over-ear fit for travel or focus."},
  "P4": {"product_name": "Language-learning course",
         "facts_line": "Structured lessons, speaking practice, feedback exercises, and weekly goals designed for steady progress."},
  "P5": {"product_name": "Subscription coffee",
         "facts_line": "Fresh beans delivered monthly, your roast preference, flexible skip/pause, and tasting notes included with each delivery."},
  "P6": {"product_name": "Productivity tool",
         "facts_line": "Task lists, calendars, reminders, and templates that sync across devices, with simple sharing for projects."},
}


## PROMPT: modular components  
This is the core prompt engineering cell. It constructs the full instructions that get sent to the GPT model for every ad generation request. Functionally it defines:

**PROMPT_HEADER** — A universal set of rules sent in every prompt that instructs the model to:
- Produce output in exactly four labelled lines: Hook, Facts, Framing, and CTA
- Stay within the defined word count range
- Reproduce the Facts line verbatim
- Avoid forbidden persuasion tactics (social proof, urgency, scarcity, authority cues, personality labels)

**Five targeting rule blocks** — One for each ad variant:
- `RULE_GENERIC` — Balanced, mainstream tone with no personality targeting
- `RULE_E_PLUS` — Tailored for high Extraversion: social cues + energetic/action language
- `RULE_E_MINUS` — Tailored for low Extraversion: calm, solo, autonomy-focused language
- `RULE_O_PLUS` — Tailored for high Openness: novelty, exploration, abstract/value framing
- `RULE_O_MINUS` — Tailored for low Openness: practical, predictable, concrete utility framing

**`build_prompt()`** — Assembles the complete prompt for a specific product + variant + paraphrase ID by combining the header, the appropriate targeting rule, and a "now generate" block that fills in the product's name and facts line.

**`build_batch_prompts()`** — A convenience wrapper that generates all variant prompts for a single product at once.


In [1]:

PROMPT_HEADER = f"""\
You are generating one advertisement stimulus for a behavioural experiment.

Output format MUST be exactly four lines:
Hook: <one sentence>
Facts: <one sentence reproduced verbatim from FACTS_LINE>
Framing: <2–3 sentences; targeting happens here>
CTA: <one neutral sentence; no urgency>

Hard constraints:
- Total combined text (Hook+Facts+Framing+CTA) must be {WORD_RANGE[0]}–{WORD_RANGE[1]} words. Count number of words before providing the message to meet the word count requirement.
- Facts line must match FACTS_LINE exactly (verbatim).
- Do NOT add any benefits/claims beyond the facts line.
- Do NOT use social proof, scarcity/urgency, or authority cues.
- Do NOT mention personality labels or address the reader as a trait (e.g., “as an extravert…”).
- Do NOT use bullet points or lists; write natural sentences.
"""

RULE_GENERIC = """\
TARGETING (GENERIC):
Write a general, broadly appealing ad that is NOT tailored to any personality trait.
Keep the framing neutral: avoid strong social/crowd language AND avoid novelty/abstract “exploration” language.
Do not imply “quiet solo recharge” either—stay balanced and mainstream.
"""

RULE_E_PLUS = """\
TARGETING (EXTRAVERSION — HIGH):
Tailor the framing to someone high on Extraversion.
Use a social orientation + energetic/action tone.
Include at least 2 social cues (e.g., friends, together, group, meet people, community) AND at least 2 high-energy/action cues (e.g., lively, buzz, kick off, jump in).
Keep cues natural (no keyword stuffing).
"""

RULE_E_MINUS = """\
TARGETING (EXTRAVERSION — LOW):
Tailor the framing to someone low on Extraversion.
Use calm, autonomy/solo tone.
Include at least 2 autonomy cues (e.g., your own pace, private, solo-friendly, personal space) AND at least 2 low-arousal cues (e.g., calm, unwind, low-key, quiet).
Avoid explicit social words like friends, crowd, meet people, together, community.
Keep cues natural (no keyword stuffing).
"""

RULE_O_PLUS = """\
TARGETING (OPENNESS — HIGH):
Tailor the framing to someone high on Openness.
Emphasize novelty/exploration + abstract/value framing + light aesthetic/creative imagery (grounded, not poetic).
Include at least 2 novelty cues (e.g., discover, new perspectives, unfamiliar, curated, experimental) AND at least 2 abstract/value cues (e.g., meaning, imagination, possibility, perspective, ideas).
Keep cues natural (no keyword stuffing).
"""

RULE_O_MINUS = """\
TARGETING (OPENNESS — LOW):
Tailor the framing to someone low on Openness.
Emphasize conventional/practical/predictable + concrete utility.
Include at least 2 predictability cues (e.g., straightforward, reliable, familiar, no surprises, proven) AND at least 2 utility cues (e.g., simple steps, efficient, clear plan, practical).
Avoid novelty/exploration language like discover, experiment, new perspectives, unexpected.
Keep cues natural (no keyword stuffing).
"""

VARIANT_TO_RULE = {
    "GENERIC": RULE_GENERIC,
    "E_PLUS": RULE_E_PLUS,
    "E_MINUS": RULE_E_MINUS,
    "O_PLUS": RULE_O_PLUS,
    "O_MINUS": RULE_O_MINUS,
}

NOW_GENERATE_BLOCK_TEMPLATE = """\
Now generate for:
PRODUCT_NAME = {product_name}
INTERNAL_VARIANT = {variant}
FACTS_LINE = {facts_line}
PARAPHRASE_ID = {paraphrase_id}
"""

def build_prompt(product_id: str, variant: str, paraphrase_id: int) -> str:
    if variant not in VARIANT_TO_RULE:
        raise ValueError(f"Unknown variant: {variant}")
    if product_id not in PRODUCTS:
        raise ValueError(f"Unknown product_id: {product_id}")

    product = PRODUCTS[product_id]
    rule_block = VARIANT_TO_RULE[variant]
    now_block = NOW_GENERATE_BLOCK_TEMPLATE.format(
        product_name=product["product_name"],
        variant=variant,
        facts_line=product["facts_line"],
        paraphrase_id=paraphrase_id
    )

    return "\n\n".join([
        textwrap.dedent(PROMPT_HEADER).strip(),
        textwrap.dedent(rule_block).strip(),
        textwrap.dedent(now_block).strip()
    ])

def build_batch_prompts(product_id: str, paraphrase_id: int) -> Dict[str, str]:
    # Returns dict: variant -> prompt
    return {v: build_prompt(product_id, v, paraphrase_id) for v in VARIANTS}

NameError: name 'WORD_RANGE' is not defined

## OUTPUT parsing + validation  

This cell defines the quality-control layer that checks every piece of text the model returns before it is accepted. It does three things:

**FORBIDDEN_PATTERNS** — A list of regular expression patterns that scan the generated text for persuasion tactics that are explicitly banned from the experiment stimuli. These include urgency phrases ("act now", "limited time"), social proof ("everyone", "most popular", "#1"), authority signals ("experts", "award"), and risk-reversal language ("guarantee"). Any ad containing these is rejected.

**`parse_four_lines()`** — Enforces that the model's raw output contains exactly four non-empty lines, each starting with the correct label (Hook, Facts, Framing, CTA). If the structure is wrong — for example the model produced five lines or skipped a label — this raises an error and triggers a retry.

**`validate_message()`** — Runs three checks on the parsed content:
1. Confirms the Facts line matches the product's defined facts line character-for-character
2. Counts the total words across all four sections and verifies the result falls within 90–110 words
3. Scans the full text for any forbidden persuasion patterns

If all checks pass, it returns the word count. If any check fails, it raises an error that triggers a retry.

In [110]:
FORBIDDEN_PATTERNS = [
    r"\blimited time\b",    # check for scarcity / artificial urgency
    r"\bact now\b",         # urgency / pressure to act immediately
    r"\bdon't miss\b",      # FOMO (fear of missing out)
    r"\bexperts?\b",        # authority / appealing to supposed expert endorsement
    r"\b#\s*1\b",           # social proof / top-ranking claim (very common in ads)
    r"\beveryone\b",        # social proof / bandwago
    r"\bmost popular\b",    # social proof / implying majority preference
    r"\baward\b",           # authority / status / credibility through external recognition
    r"\bguarantee\b",       # risk reversal / removing perceived purchase risk
    r"\bonly today\b",      # scarcity / extreme time limitation
    r"\brated\b.*\bstars\b",# social proof / quantified rating appeal
]

def word_count(s: str) -> int:
    return len(re.findall(r"\b[\w’'-]+\b", s))

def parse_four_lines(text: str) -> Dict[str, str]:
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]
    if len(lines) != 4:
        raise ValueError(f"Expected exactly 4 non-empty lines, got {len(lines)}.\nRaw:\n{text}")

    def get(prefix: str) -> str:
        for ln in lines:
            if ln.startswith(prefix):
                return ln[len(prefix):].strip()
        raise ValueError(f"Missing required line prefix: {prefix}\nRaw:\n{text}")

    return {
        "hook": get("Hook:"),
        "facts": get("Facts:"),
        "framing": get("Framing:"),
        "cta": get("CTA:")
    }

def validate_message(parts: Dict[str, str], facts_line: str) -> Tuple[int, List[str]]:
    
    # facts line verbatim check
    if parts["facts"].strip() != facts_line.strip():
        raise ValueError("Facts line mismatch (must match FACTS_LINE verbatim).")

    full = f'{parts["hook"]} {parts["facts"]} {parts["framing"]} {parts["cta"]}'.strip()
    wc = word_count(full)
    if not (WORD_RANGE[0] <= wc <= WORD_RANGE[1]):
        raise ValueError(f"Word count {wc} outside range {WORD_RANGE}.")

    low = full.lower()
    forbidden_hits = [pat for pat in FORBIDDEN_PATTERNS if re.search(pat, low)]
    if forbidden_hits:
        raise ValueError(f"Forbidden cue(s) detected: {forbidden_hits}")

    return wc, forbidden_hits

## Record format (JSON schema)  
**Unique ID Generation via Hashing**  
This cell establishes a reproducible, collision-resistant system for uniquely identifying every generated advertisement. It creates two types of IDs:

**`make_condition_id()`** — Produces a stable 16-character ID that identifies the *experimental condition* (the combination of product, variant, paraphrase number, facts line, and word range). This ID will be the same every time you run the notebook for the same inputs — useful for tracking which conditions have already been generated.

**`make_output_id()`** — Produces a 16-character ID that identifies a *specific generated output* (combining the condition ID with the model name, temperature, and the actual message text). Because the message text is part of the hash, two different outputs for the same condition will always have different output IDs even if all settings are identical.

Both IDs are derived by hashing a canonical string representation of the input parameters using SHA-256, then taking the first 16 characters. This gives the saved JSON files unique, deterministic filenames.

In [37]:
import hashlib
from datetime import datetime, timezone

def sha16(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def make_condition_id(product_id, variant, paraphrase_id, facts_line, word_range) -> str:
    canonical = f"{product_id}|{variant}|p{paraphrase_id}|wr={word_range}|facts={facts_line}"
    return sha16(canonical)

def make_output_id(condition_id, model, temperature, full_message) -> str:
    canonical = f"{condition_id}|{model}|temp={temperature}|msg={full_message}"
    return sha16(canonical)

## Record Assembly (`make_record`)  

This cell defines the function that packages a successfully validated advertisement into a structured JSON record ready to be saved to disk. It acts as the "assembly line final step," taking all the pieces produced earlier and organising them into a single well-structured document.

The record it produces contains six top-level sections:
- **id** — The unique condition ID and output ID for traceability
- **generation** — Which model was used, at what temperature, and the UTC timestamp of creation
- **meta** — The product details, the personality trait axis targeted (EXTRAVERSION, OPENNESS, or NONE), the variant name, the paraphrase number, and the word target/range
- **constraints** — A set of boolean flags documenting the rules that were applied (facts must match verbatim, no personality labels, no extra claims, etc.) — useful for auditing the dataset later
- **full_prompt** — The exact prompt that was sent to the model, stored for full reproducibility
- **content** — The four individual ad fields (Hook, Facts, Framing, CTA) plus the assembled full message
- **self_checks** — The actual word count, verbatim facts confirmation, and any forbidden pattern hits

In [63]:
def make_record(product_id: str,
                variant: str,
                paraphrase_id: int,
                parts: Dict[str, str],
                wc: int,
                forbidden_hits: List[str],
                model,
                temperature, 
                prompt) -> Dict:
                
    
    product = PRODUCTS[product_id]
    trait_axis = "EXTRAVERSION" if variant.startswith("E_") else ("OPENNESS" if variant.startswith("O_") else "NONE")

    full_message = f'{parts["hook"]} {parts["facts"]} {parts["framing"]} {parts["cta"]}'.strip()

    # IDs
    condition_id = make_condition_id(
        product_id=product_id,
        variant=variant,
        paraphrase_id=paraphrase_id,
        facts_line=product["facts_line"],
        word_range=WORD_RANGE,
    )
    output_id = make_output_id(
        condition_id=condition_id,
        model=model,
        temperature=temperature,
        full_message=full_message,
    )
    
    full_prompt = prompt
    
    return {    
        "id": {
            "condition_id": condition_id,
            "output_id": output_id
        },
        "generation": {
            "model": model,
            "temperature": temperature,
            "timestamp_utc": utc_now_iso()
        },
        "meta": {
            "product_id": product_id,
            "product_name": product["product_name"],
            "trait_axis": trait_axis,
            "variant": variant,
            "paraphrase_id": paraphrase_id,
            "word_target": 100,
            "word_range": [WORD_RANGE[0], WORD_RANGE[1]] # range : 90 - 110
        },
        "constraints": {
            "facts_line_must_match_verbatim": True,
            "no_personality_labels": True,
            "no_extra_benefits_or_claims": True,
            "no_social_proof_scarcity_authority": True,
            "cta_strength_equal_across_variants": True
        },
        "full_prompt": full_prompt,
        "content": {
            "hook": parts["hook"],
            "facts": parts["facts"],
            "framing": parts["framing"],
            "cta": parts["cta"],
            "full_message": full_message
        },
        "self_checks": {
            "approx_word_count_full_message": wc,
            "facts_line_verbatim_confirmed": True,
            "forbidden_items_present": forbidden_hits
        }
    }

## Helper functions, API Call, Generation Pipeline + retry logic & File Save  
 
This cell contains the three functions that execute the end-to-end generation workflow for a single advertisement:

**`call_model()`** — Sends the assembled prompt to the OpenAI API and returns the raw text response from the model. It uses the model and temperature settings defined earlier.

**`generate_one_message()`** — Orchestrates the full generation-and-validation pipeline for one specific product/variant/paraphrase combination. It:
1. Builds the prompt
2. Calls the model to get a candidate response
3. Parses the four-line structure
4. Validates word count, verbatim facts, and forbidden patterns
5. Packages everything into a record

If any step fails (because the model produced malformed output, wrong word count, or used a forbidden phrase), it automatically retries up to 7 times with a small increasing delay between attempts — necessary because GPT outputs are non-deterministic and may occasionally violate constraints.

**`save_record()`** — Takes a completed, validated record and writes it to disk as a JSON file in the output directory(stimuli_out). The filename encodes the product ID, variant, paraphrase number, model name, and unique output ID so files are human-readable and won't overwrite each other.

In [134]:
FAIL_DIR = Path("stimuli_out_fail")

def call_model(prompt: str, client: OpenAI) -> str:
    resp = client.responses.create(
        model=MODEL,
        input=prompt,
        temperature=TEMPERATURE,
    )
    return (resp.output_text or "").strip()

def save_fail_record(product_id: str,
                     variant: str,
                     paraphrase_id: int,
                     raw_text: str,
                     error: str,
                     attempt: int,
                     prompt: str,
                     parts: Dict = None) -> Path:
    '''
    Save a failed generation attempt to stimuli_out_fail/ so we don't lose
    paid API outputs that narrowly miss validation (e.g. word count just outside range).
    Includes whatever could be extracted (raw text + partial parts if parsing succeeded).
    '''
    FAIL_DIR.mkdir(parents=True, exist_ok=True)
    product = PRODUCTS[product_id]

    # Build a lightweight fail record
    record = {
        "status": "FAILED_VALIDATION",
        "error": str(error),
        "attempt": attempt,
        "generation": {
            "model": MODEL,
            "temperature": TEMPERATURE,
            "timestamp_utc": utc_now_iso()
        },
        "meta": {
            "product_id": product_id,
            "product_name": product["product_name"],
            "variant": variant,
            "paraphrase_id": paraphrase_id,
            "word_range": [WORD_RANGE[0], WORD_RANGE[1]]
        },
        "full_prompt": prompt,
        "raw_output": raw_text,
        # If parsing succeeded before validation failed, save the extracted parts too
        "parsed_parts": parts if parts is not None else None
    }

    # Add word count if we have full text to count
    if parts is not None:
        full = f'{parts["hook"]} {parts["facts"]} {parts["framing"]} {parts["cta"]}'.strip()
        record["approx_word_count"] = word_count(full)
        record["full_message"] = full

    # Unique filename: product_variant_p{id}_attempt{n}_timestamp
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    fname = f'{product_id}_{variant}_p{paraphrase_id}_attempt{attempt}_{MODEL}_{ts}.json'
    path = FAIL_DIR / fname
    path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def generate_one_message(product_id: str,
                         variant: str,
                         paraphrase_id: int,
                         client: OpenAI,
                         max_retries: int = 10) -> Dict:
    '''
    Generation pipeline:
    1) call_model(): sample a candidate message from the LLM
    2) parse_four_lines(): enforce the required output format (exactly 4 lines/fields)
    3) validate_message(): enforce content constraints:
       - word count window
       - variant-specific cue requirements (E± / O± dials)
       - facts consistency using the product's facts_line
    If parsing/validation fails, retry (up to max_retries) because the model output is stochastic.
    '''

    prompt = build_prompt(product_id, variant, paraphrase_id)
    facts_line = PRODUCTS[product_id]["facts_line"]

    last_err = None
    for attempt in range(1, max_retries + 1):
        raw = call_model(prompt, client)
        parts = None
        try:
            parts = parse_four_lines(raw)
            wc, hits = validate_message(parts, facts_line)
            return make_record(product_id, variant, paraphrase_id, parts, wc, hits, model=MODEL, temperature=TEMPERATURE, prompt=prompt)
        except Exception as e:
            last_err = e
            # Save the failed attempt so we don't lose the paid output
            fail_path = save_fail_record(
                product_id=product_id,
                variant=variant,
                paraphrase_id=paraphrase_id,
                raw_text=raw,
                error=e,
                attempt=attempt,
                prompt=prompt,
                parts=parts  # None if parse failed, dict if validate failed
            )
            print(f"    [attempt {attempt} failed: {e}] -> saved to {fail_path.name}")
            time.sleep(0.5 * attempt)

    raise RuntimeError(f"Failed after {max_retries} retries for {product_id}/{variant}/p{paraphrase_id}: {last_err}")

def save_record(record: Dict) -> Path:
    '''
    Make the directory with unique filename based on condition and output IDs, then save the record as JSON.
    '''
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    m = record["meta"]
    oid = record["id"]["output_id"]     # unique per output
    model = record["generation"]["model"].replace("/", "-")


    fname = f'{m["product_id"]}_{m["variant"]}_p{m["paraphrase_id"]}_{model}_{oid}.json'
    path = OUT_DIR / fname

    path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

## Generate 1 stimulus  
This cell defines a convenience function for generating and inspecting **one advertisement** at a time. It's designed for testing and spot-checking individual combinations before running the full batch.

When called, it:
1. Verifies the OpenAI API key is available in the environment
2. Creates an authenticated OpenAI client
3. Calls `generate_one_message()` to produce and validate a single ad
4. Saves the resulting JSON record to disk
5. Prints a human-readable preview showing:
   - The file path and word count of the saved record
   - The full assembled advertisement as a single paragraph
   - Each of the four component fields (Hook, Facts, Framing, CTA) printed separately for easy inspection

This is particularly useful for verifying that targeting cues are working correctly before committing to a full multi-product, multi-variant batch run.

In [114]:
def single_stim(product_id, variant, pid):
    if "OPENAI_API_KEY" not in os.environ:
        raise EnvironmentError("Set OPENAI_API_KEY in your environment first.")

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    record = generate_one_message(product_id, variant, pid, client=client)
    path = save_record(record)

    print(f"Saved: {path}  (words={record['self_checks']['approx_word_count_full_message']})")

    # show the output message
    print("\n--- MODEL OUTPUT (full_message) ---")
    print(record["content"]["full_message"])

    # optional: show the 4 lines separately
    print("\n--- split fields ---")
    print("Hook:", record["content"]["hook"])
    print("Facts:", record["content"]["facts"])
    print("Framing:", record["content"]["framing"])
    print("CTA:", record["content"]["cta"])

    return record, path

## Example: Run a Single Stimulus

In [99]:
record, path = single_stim("P2", "O_PLUS", 1) # GENERIC, E_PLUS, E_MINUS, O_PLUS, O_MINUS
# print(record["content"]["full_message"])

Saved: stimuli_out\P2_O_PLUS_p1_gpt-4o_ca789f02491f760a.json  (words=100)

--- MODEL OUTPUT (full_message) ---
Discover new paths to a balanced life with our innovative fitness app. Personalized workouts from 10–30 minutes, progress tracking, and adjustable plans for strength, cardio, or mobility. Explore the possibilities of your fitness journey, where each session is crafted to inspire your imagination and elevate your routine. Designed for those who seek new perspectives and value the fusion of creativity with wellness, this app opens a gateway to an unfamiliar realm of exercise. Embrace the art of self-discovery through experimental workouts that align with your personal vision of health. Experience a fresh take on fitness with this app today.

--- split fields ---
Hook: Discover new paths to a balanced life with our innovative fitness app.
Facts: Personalized workouts from 10–30 minutes, progress tracking, and adjustable plans for strength, cardio, or mobility.
Framing: Explore th

## Scan helper  

Scan Existing Output Files (`load_completed_keys`)  
This cell defines a utility function that scans the output directory for JSON files that have already been successfully generated, so the main batch runner can skip them on subsequent runs.

For every JSON file it finds, it reads the metadata and constructs a job key in the format `{product_id}__{variant}__p{paraphrase_id}__{model}`. All these keys are collected into a set, which is then returned.

This is what makes the batch generation **resumable** — if the notebook is interrupted mid-run (e.g., due to API rate limits or a crash), re-running it will pick up exactly where it left off rather than re-generating and overwriting already completed stimuli. The function also prints a summary showing how many files were scanned, successfully parsed, and skipped due to errors.

In [127]:
import json
from pathlib import Path

def load_completed_keys(out_dir: Path) -> set[str]:
    """
    Scan out_dir for existing JSON records and return a set of completed job keys.

    Key format (now includes model):
        {product_id}__{variant}__p{paraphrase_id}__{model}

    Prints a short summary so you can verify scanning worked.
    """
    keys = set()
    scanned = 0
    parsed_ok = 0
    skipped = 0

    for fp in out_dir.glob("*.json"):
        scanned += 1
        try:
            rec = json.loads(fp.read_text(encoding="utf-8"))
            m = rec["meta"]
            product_id = m["product_id"]
            variant = m["variant"]
            pid = m["paraphrase_id"]

            # model stored in generation block (per your save_record)
            model = rec.get("generation", {}).get("model", "UNKNOWN")

            keys.add(f"{product_id}__{variant}__p{pid}__{model}")
            parsed_ok += 1
        except Exception:
            skipped += 1
            continue

    print(f"[load_completed_keys] dir={out_dir.resolve()}")
    print(f"[load_completed_keys] scanned={scanned} files | parsed={parsed_ok} | skipped={skipped} | unique_keys={len(keys)}")
    return keys

### Main Batch Generation Loop (`main`)
This cell defines the primary orchestration function that generates the **complete set of advertisement stimuli** across all products, variants, and paraphrase IDs.

Its logic works as follows:
1. Authenticates with the OpenAI API
2. Calls `load_completed_keys()` to find all already-generated stimuli so they can be skipped
3. Iterates over every product (P1–P6), every targeting variant (GENERIC, E_PLUS, E_MINUS, O_PLUS, O_MINUS), and every paraphrase ID (1, 2, 3)
4. For the GENERIC variant specifically, only one paraphrase is generated per product (since a generic ad doesn't need multiple personality-targeted variants)
5. For each combination not yet completed, calls `generate_one_message()` and `save_record()`, then immediately marks it as complete in memory to prevent duplicates even within the same run
6. Prints progress throughout, and finishes with a summary of how many new messages were generated

The full matrix produces up to **78 advertisements** (6 products × 4 personality variants × 3 paraphrases + 6 generic = 54 + 6 = 60... though the exact count depends on the PARAPHRASE_IDS setting).

In [146]:
def main():
    if "OPENAI_API_KEY" not in os.environ:
        raise EnvironmentError("Set OPENAI_API_KEY in your environment first.")

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    total = 0
    variants = list(VARIANT_TO_RULE.keys())  # <-- use existing mapping

    completed = load_completed_keys(OUT_DIR)
    
        


    for product_id in PRODUCTS.keys():
        print(f"\n== Product {product_id} ==")

        for variant in variants:
            # Only 1 GENERIC per product; all others use all paraphrase IDs
            pids = [PARAPHRASE_IDS[0]] if variant == "GENERIC" else PARAPHRASE_IDS

            for pid in pids:
                job_key = f"{product_id}__{variant}__p{pid}__{MODEL}"    
                if job_key in completed:
                    print(f"  Skipping already completed: {job_key}")
                    continue
                print(f"  -> variant={variant} paraphrase={pid}")

                record = generate_one_message(product_id, variant, pid, client=client)
                path = save_record(record)
                total += 1
                completed.add(job_key)  # <-- IMPORTANT: add to completed set immediately after saving to avoid duplicates on reruns
                print(f"Saved: {path} (words={record['self_checks']['approx_word_count_full_message']})")

    print(f"\nDone. Generated {total} messages into {OUT_DIR.resolve()}")

### Execute Full Batch Generation

In [160]:
main()

[load_completed_keys] dir=C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\Code\stimuli_out
[load_completed_keys] scanned=105 files | parsed=105 | skipped=0 | unique_keys=91

== Product P1 ==
  Skipping already completed: P1__GENERIC__p1__gpt-4o
  Skipping already completed: P1__E_PLUS__p1__gpt-4o
  Skipping already completed: P1__E_PLUS__p2__gpt-4o
  Skipping already completed: P1__E_PLUS__p3__gpt-4o
  Skipping already completed: P1__E_MINUS__p1__gpt-4o
  Skipping already completed: P1__E_MINUS__p2__gpt-4o
  Skipping already completed: P1__E_MINUS__p3__gpt-4o
  Skipping already completed: P1__O_PLUS__p1__gpt-4o
  Skipping already completed: P1__O_PLUS__p2__gpt-4o
  Skipping already completed: P1__O_PLUS__p3__gpt-4o
  Skipping already completed: P1__O_MINUS__p1__gpt-4o
  Skipping already completed: P1__O_MINUS__p2__gpt-4o
  Skipping already completed: P1__O_MINUS__p3__gpt-4o

== Product P2 ==
  Skipping already completed: P2__GENERIC_

# Empath

# Stimulus Analysis Pipeline (Empath + Anchor Similarity + Sign Tests)

This notebook:

1. Reads raw stimuli JSON files from `stimuli_out/`
2. Appends analysis fields:
   - `analysis.empath_bundles.*`
   - `analysis.anchor_similarity.*` (top match, margins, correctness)
   - `analysis.nrc_emotions.profile` (optional)
3. Writes analysed copies to `stimuli_out_analysed/` (same filenames)
4. Produces pre-test summary tables:
   - Empath bundle sign tests across products (E_PLUS vs E_MINUS; O_PLUS vs O_MINUS)
   - Anchor alignment accuracy + axis-margin sign tests


### Cell 1 — Setup, Analysis Pipeline, and File Output

This is the primary execution cell. It sets up every component required for the three-layer analysis, runs that analysis across all generated advertisement files, writes the enriched output to disk, and loads the results into memory for the statistics cell that follows.

**Libraries and paths**

The cell begins by importing all required libraries: standard tools for file handling, JSON processing, and numerical operations; Empath for lexical category scoring; the sentence-transformers library for semantic embedding; scikit-learn's cosine similarity function; and NRCLex for emotion profiling. It then defines the input folder (`stimuli_out/`) and the output folder (`stimuli_out_analysed/`), creating the output folder if it does not already exist, and verifies the input folder is present before proceeding.

**Helper functions**

Two utility functions are defined that are used throughout the rest of the cell. `get_nested()` safely retrieves a value from a deeply nested dictionary by following a sequence of keys, returning a specified default rather than raising an error if any key is missing. `set_nested()` writes a value into a nested location within a dictionary specified as a dot-separated path string such as `"analysis.empath_bundles.E_social"`, creating any missing intermediate dictionaries automatically.

**Layer 1 — Empath bundle analysis**

The Empath lexicon is initialised and all 194 valid category keys are collected into a set. The six analysis bundles are then defined in `BUNDLES_RAW`: two for the Extraversion axis (E_social capturing social/interpersonal language; E_energy capturing high-arousal positive language) and four for the Openness axis (O_creative for aesthetic language; O_abstract for intellectual language; O_practical for transactional/utilitarian language; O_novelty for exploration and travel language). These categories are aligned with the framing dimensions described in Hirsh, Kang and Bodenhausen (2012) and Matz et al. (2024).

Each bundle then passes through an audit step: every category name is checked against the set of valid Empath keys, and any name that does not match is silently dropped with a warning printed to the console. This directly addresses a known flaw in earlier versions of the code where invalid names like `zest` scored zero without warning, diluting bundle means. The validated bundles are stored in `BUNDLES` and the audit results in `AUDIT`. `empath_scores()` runs the Empath analysis with normalisation enabled so scores are expressed as rates per token, making them comparable across messages of different lengths. `bundle_mean()` averages the scores across the valid categories in a bundle, returning NaN rather than zero if no valid categories remain to avoid ambiguity.

**Layer 2 — Sentence-transformer anchor similarity**

The `all-MiniLM-L6-v2` sentence-transformer model is loaded. This is a compact, freely available model that runs locally without an API call. Four anchor sentences are defined — one for each targeted personality variant. These are short, topic-neutral descriptions of each personality framing written to reflect the characteristics described in the personalised persuasion literature. The embeddings for all four anchors are pre-computed once outside the loop so they are not recomputed for every message.

The `anchor_feats()` function takes a message text, encodes it using the same model, and computes the cosine similarity between the message embedding and each of the four anchor embeddings. It returns six values: the raw similarity score to each anchor, the label of the anchor with the highest similarity (`top_match`), the margin between the top score and the second-highest score (`alignment_margin`), and two axis-specific margins — E_margin is the E_PLUS similarity minus the E_MINUS similarity, and O_margin is the O_PLUS similarity minus the O_MINUS similarity. A positive E_margin on an E_PLUS message confirms the message is semantically closer to high-Extraversion framing than low-Extraversion framing; a negative E_margin on an E_MINUS message confirms the opposite. These margins are the primary continuous signals for Layer 2 statistics.

**Layer 3 — NRC emotion profiles**

The `nrc_profile()` function runs NRC Emotion Lexicon analysis on a text string, counting word occurrences across eight emotion categories (joy, anticipation, trust, surprise, anger, fear, sadness, disgust) plus positive and negative sentiment, then normalises the counts by the total so results are expressed as proportions comparable across different message lengths.

**Main analysis loop**

The cell loops over every JSON file in `stimuli_out/`. For each file it handles both single-record files (stored as a JSON object) and multi-record files (stored as a JSON array). For each record it extracts the full message text, the variant label, the product ID, and the model name. If the text is missing or empty the record is skipped and counted. Otherwise all three analysis layers run in sequence and their outputs are written into the record under `analysis.empath_bundles`, `analysis.empath_bundle_meta`, `analysis.anchor_similarity`, and `analysis.nrc_emotions.profile`. The `correct_alignment` flag inside `anchor_similarity` is set to True if the top anchor match equals the intended variant, False if not, and None for GENERIC messages which have no targeted anchor. Each record is also appended to the `records` list in memory — keyed by filename, product ID, variant, and model — so the statistics cell can work directly from memory without re-reading the files. The enriched file is then written to `stimuli_out_analysed/` under the same filename. A summary is printed at the end showing how many files were written and how many records were loaded into memory.

In [180]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from empath import Empath
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nrclex import NRCLex

# -------- Paths --------
IN_DIR = Path("stimuli_out")
OUT_DIR = Path("stimuli_out_analysed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert IN_DIR.exists(), f"Missing input folder: {IN_DIR.resolve()}"

TEXT_PATH = ("content", "full_message")

# -------- Helpers --------
def get_nested(d, path, default=None):
    cur = d
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def set_nested(d, dotted_path, value):
    keys = dotted_path.split(".")
    cur = d
    for k in keys[:-1]:
        if k not in cur or not isinstance(cur[k], dict):
            cur[k] = {}
        cur = cur[k]
    cur[keys[-1]] = value

# -------- Layer 1: Empath bundles (audited) --------
lex = Empath()
VALID = set(lex.cats.keys())

BUNDLES_RAW = {
    # Extraversion axis
    "E_social":   ["friends", "meeting", "communication", "party", "celebration", "social_media"],
    "E_energy":   ["positive_emotion", "joy", "cheerfulness", "anticipation", "fun"],
    # Openness axis
    "O_creative": ["art", "music", "beauty", "fashion", "hipster"],
    "O_abstract": ["philosophy", "science", "technology"],
    "O_practical":["work", "business", "office", "money", "payment", "economics", "shopping"],
    "O_novelty":  ["surprise", "traveling", "tourism", "vacation"],
}

BUNDLES = {}
AUDIT = {}
for b, cats in BUNDLES_RAW.items():
    kept = [c for c in cats if c in VALID]
    dropped = [c for c in cats if c not in VALID]
    BUNDLES[b] = kept
    AUDIT[b] = {"kept": kept, "dropped": dropped}

print("Empath audit (dropped categories):")
for b, info in AUDIT.items():
    print(f"  {b}: dropped={info['dropped']}" if info["dropped"] else f"  {b}: OK ({len(info['kept'])} cats)")

def empath_scores(text: str) -> dict:
    return lex.analyze(text, normalize=True)

def bundle_mean(scores: dict, cats: list[str]) -> float:
    if not cats:
        return float("nan")
    vals = [float(scores.get(c, 0.0)) for c in cats]
    return float(sum(vals) / len(vals))

# -------- Layer 2: Anchor similarity --------
st_model = SentenceTransformer("all-MiniLM-L6-v2")

# Replace these with your Matz-derived topic-neutral anchors when you finalize them.
ANCHORS = {
    "E_PLUS":  "A lively, exciting experience where you can connect with friends, meet new people, and jump into the buzz together.",
    "E_MINUS": "A calm, low-key experience you can enjoy at your own pace—private, spacious, and comfortable without crowds or pressure.",
    "O_PLUS":  "Discover something new and explore fresh perspectives—an imaginative, curiosity-driven experience rich in ideas and aesthetic detail.",
    "O_MINUS": "A straightforward, reliable option with familiar structure—clear steps, efficient planning, and proven choices with no surprises.",
}
anchor_keys = list(ANCHORS.keys())
anchor_emb = st_model.encode([ANCHORS[k] for k in anchor_keys], normalize_embeddings=True)

def anchor_feats(text: str) -> dict:
    emb = st_model.encode([text], normalize_embeddings=True)
    sims = cosine_similarity(emb, anchor_emb)[0]
    scores = {k: float(s) for k, s in zip(anchor_keys, sims)}
    top = max(scores, key=scores.get)
    sorted_s = sorted(scores.values(), reverse=True)
    margin = float(sorted_s[0] - sorted_s[1]) if len(sorted_s) >= 2 else 0.0
    return {
        "scores": scores,
        "top_match": top,
        "alignment_margin": margin,
        "axis_margins": {
            "E_margin": float(scores["E_PLUS"] - scores["E_MINUS"]),
            "O_margin": float(scores["O_PLUS"] - scores["O_MINUS"]),
        },
    }

# -------- Layer 3: NRC (optional, but cheap + useful triangulation) --------
def nrc_profile(text: str) -> dict:
    emo = NRCLex(text)
    total = sum(emo.raw_emotion_scores.values()) or 1
    return {k: float(v/total) for k, v in emo.raw_emotion_scores.items()}

# -------- Run analysis on all stimuli_out files, write to stimuli_out_analysed --------
files = sorted(IN_DIR.glob("*.json"))
print(f"Found {len(files)} files in {IN_DIR.resolve()}")

records = []  # for summary stats later

updated = 0
skipped = 0

for fp in files:
    raw = json.loads(fp.read_text(encoding="utf-8"))
    is_single = isinstance(raw, dict)
    recs = [raw] if is_single else raw

    for rec in recs:
        text = get_nested(rec, TEXT_PATH)
        if not isinstance(text, str) or not text.strip():
            skipped += 1
            continue

        variant = get_nested(rec, ("meta","variant"))
        product_id = get_nested(rec, ("meta","product_id"))
        model = get_nested(rec, ("generation","model"))

        # Empath bundles (+ audit metadata)
        scores = empath_scores(text)
        for b, cats in BUNDLES.items():
            set_nested(rec, f"analysis.empath_bundles.{b}", bundle_mean(scores, cats))
            set_nested(rec, f"analysis.empath_bundle_meta.{b}.kept_categories", AUDIT[b]["kept"])
            set_nested(rec, f"analysis.empath_bundle_meta.{b}.dropped_categories", AUDIT[b]["dropped"])

        # Anchor similarity
        af = anchor_feats(text)
        set_nested(rec, "analysis.anchor_similarity.scores", af["scores"])
        set_nested(rec, "analysis.anchor_similarity.top_match", af["top_match"])
        set_nested(rec, "analysis.anchor_similarity.alignment_margin", af["alignment_margin"])
        set_nested(rec, "analysis.anchor_similarity.axis_margins", af["axis_margins"])
        if variant in ANCHORS:
            set_nested(rec, "analysis.anchor_similarity.correct_alignment", af["top_match"] == variant)
        else:
            set_nested(rec, "analysis.anchor_similarity.correct_alignment", None)

        # NRC
        set_nested(rec, "analysis.nrc_emotions.profile", nrc_profile(text))

        records.append({
            "filename": fp.name,
            "product_id": product_id,
            "variant": variant,
            "model": model,
            "rec": rec,
        })

    out_path = OUT_DIR / fp.name
    out_path.write_text(json.dumps(recs[0] if is_single else recs, ensure_ascii=False, indent=2), encoding="utf-8")
    updated += 1

print(f"Done. Wrote {updated} analysed files to {OUT_DIR.resolve()}. Skipped {skipped} records with missing text.")
print(f"Loaded {len(records)} analysed records into `records` for summary statistics.")


Empath audit (dropped categories):
  E_social: OK (6 cats)
  E_energy: OK (5 cats)
  O_creative: OK (5 cats)
  O_abstract: OK (3 cats)
  O_practical: OK (7 cats)
  O_novelty: OK (4 cats)
Found 78 files in C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\Code\stimuli_out
Done. Wrote 78 analysed files to C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\Code\stimuli_out_analysed. Skipped 0 records with missing text.
Loaded 78 analysed records into `records` for summary statistics.


### Cell 2 — Summary Statistics: Sign Tests and Alignment Accuracy

This cell takes the `records` list populated by Cell 1 and produces the three summary tables needed to report our pre-test validation results. All computation is in memory so this cell can be re-run quickly without re-running the full analysis.

**Building the analysis dataframe**

The cell begins by flattening every record into a single row containing the product ID, variant, model, all six Empath bundle scores, the two axis margins (E_margin and O_margin), and the correct-alignment flag. This produces a tidy dataframe `df` with one row per generated advertisement, which all subsequent statistics draw from.

**Empath bundle sign tests (Layer 1 statistics)**

The `product_level_means()` helper computes the mean bundle score for each product separately for a given variant, returning a series indexed by product ID. `sign_test_across_products()` then takes two variants and a bundle, finds the products that appear in both variants, computes the per-product difference in mean bundle score between the two variants, counts how many products show a difference in the expected direction, and runs a one-sided binomial test on that count against a null of 0.5.

This is the statistically appropriate test at our sample size. With only three paraphrase versions per variant per product, computing effect sizes like Cohen's d would produce unreliable estimates with enormous confidence intervals. The sign test instead asks a simpler and more honest question: across our six products, in how many cases did the targeted variant outscore the control variant on the relevant bundle? If E_PLUS scores higher than E_MINUS on E_social in all six products the probability of that occurring by chance is 0.5 to the power of 6, which is 1.6%.

Six tests are run in total. For the Extraversion axis: E_PLUS versus E_MINUS on E_social, and E_PLUS versus E_MINUS on E_energy, both expecting E_PLUS higher. For the Openness axis: O_PLUS versus O_MINUS on O_creative, O_abstract, and O_novelty, all expecting O_PLUS higher. The O_practical test is run in the reversed direction — O_MINUS versus O_PLUS — because high-Openness framing is specifically designed to avoid practical and transactional language, so O_PLUS messages are expected to score lower on O_practical than O_MINUS messages. The results are displayed as a clean dataframe showing the contrast direction, bundle name, number of products tested, number of products where the expected direction held, and the binomial p-value.

**Anchor alignment accuracy (Layer 2 statistics, part 1)**

From the full dataframe, GENERIC messages are filtered out since they have no intended anchor. From the remaining targeted messages — those with variants E_PLUS, E_MINUS, O_PLUS, or O_MINUS — the cell counts how many have `correct_alignment` equal to True and runs a binomial test against a null hypothesis of 0.25. The chance baseline is 0.25 rather than 0.5 because there are four possible anchors, so random matching would be correct one quarter of the time. A p-value well below 0.05 here means our messages are semantically distinguishable by personality framing at a rate significantly above chance, providing evidence that the LLM-generated text encodes the intended targeting in a way that goes beyond surface word choice.

The result is displayed as a single-row summary showing the total number of targeted messages, the number correctly aligned, the accuracy proportion, and the binomial p-value.

**Axis margin sign tests (Layer 2 statistics, part 2)**

`axis_margin_sign_test()` applies the same product-level binomial sign test logic used for Empath, but to the continuous axis margin scores rather than bundle means. For the E_margin test, it checks whether E_PLUS messages have a higher mean E_margin than E_MINUS messages across products — meaning E_PLUS messages are more consistently oriented toward the high-Extraversion anchor relative to the low-Extraversion anchor. The equivalent test for O_margin compares O_PLUS against O_MINUS messages. Two tests are run and the results are displayed as a table, giving you a second, semantically grounded confirmation of directional alignment that is independent of the lexical Empath analysis.

In [193]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from empath import Empath
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nrclex import NRCLex

# ============================================================
# CONFIG: analyse one or multiple model runs
# - Each run reads raw JSON from in_dir
# - Writes analysed copies (same filenames) to out_dir
# - Keeps raw inputs untouched
# ============================================================

RUNS = [
    # Main model run
    {"name": "gpt_main", "in_dir": Path("stimuli_out"), "out_dir": Path("stimuli_out_analysed")},
    # Optional second model run (edit paths to your repo structure)
    {"name": "gpt_nano", "in_dir": Path("stimuli_out/nano"), "out_dir": Path("stimuli_out_analysed/nano")},
]

TEXT_PATH = ("content", "full_message")

# ---------------- Helpers ----------------
def get_nested(d, path, default=None):
    cur = d
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def set_nested(d, dotted_path, value):
    keys = dotted_path.split(".")
    cur = d
    for k in keys[:-1]:
        if k not in cur or not isinstance(cur[k], dict):
            cur[k] = {}
        cur = cur[k]
    cur[keys[-1]] = value

# ============================================================
# Layer 1 (Q1): Empath bundles with category audit
# - Prevents silent zeros from invalid category names
# ============================================================
lex = Empath()
VALID = set(lex.cats.keys())

BUNDLES_RAW = {
    # Extraversion axis
    "E_social":   ["friends", "meeting", "communication", "party", "celebration", "social_media"],
    "E_energy":   ["positive_emotion", "joy", "cheerfulness", "anticipation", "fun"],
    # Openness axis
    "O_creative": ["art", "music", "beauty", "fashion", "hipster"],
    "O_abstract": ["philosophy", "science", "technology"],
    "O_practical":["work", "business", "office", "money", "payment", "economics", "shopping"],
    "O_novelty":  ["surprise", "traveling", "tourism", "vacation"],
}

BUNDLES = {}
AUDIT = {}
for b, cats in BUNDLES_RAW.items():
    kept = [c for c in cats if c in VALID]
    dropped = [c for c in cats if c not in VALID]
    BUNDLES[b] = kept
    AUDIT[b] = {"kept": kept, "dropped": dropped}

print("Empath audit (dropped categories):")
for b, info in AUDIT.items():
    print(f"  {b}: dropped={info['dropped']}" if info["dropped"] else f"  {b}: OK ({len(info['kept'])} cats)")

def empath_scores(text: str) -> dict:
    return lex.analyze(text, normalize=True)

def bundle_mean(scores: dict, cats: list[str]) -> float:
    if not cats:
        return float("nan")
    vals = [float(scores.get(c, 0.0)) for c in cats]
    return float(sum(vals) / len(vals))

# ============================================================
# Layer 2 (Q2): Anchor similarity (sentence-transformers)
# - Swap these anchors for your final Matz-derived topic-neutral anchors
# ============================================================
st_model = SentenceTransformer("all-MiniLM-L6-v2")

ANCHORS = {
    "E_PLUS":  "A lively, exciting experience where you can connect with friends, meet new people, and jump into the buzz together.",
    "E_MINUS": "A calm, low-key experience you can enjoy at your own pace—private, spacious, and comfortable without crowds or pressure.",
    "O_PLUS":  "Discover something new and explore fresh perspectives—an imaginative, curiosity-driven experience rich in ideas and aesthetic detail.",
    "O_MINUS": "A straightforward, reliable option with familiar structure—clear steps, efficient planning, and proven choices with no surprises.",
}
anchor_keys = list(ANCHORS.keys())
anchor_emb = st_model.encode([ANCHORS[k] for k in anchor_keys], normalize_embeddings=True)

def anchor_feats(text: str) -> dict:
    emb = st_model.encode([text], normalize_embeddings=True)
    sims = cosine_similarity(emb, anchor_emb)[0]
    scores = {k: float(s) for k, s in zip(anchor_keys, sims)}
    top = max(scores, key=scores.get)
    sorted_s = sorted(scores.values(), reverse=True)
    margin = float(sorted_s[0] - sorted_s[1]) if len(sorted_s) >= 2 else 0.0
    return {
        "scores": scores,
        "top_match": top,
        "alignment_margin": margin,
        "axis_margins": {
            "E_margin": float(scores["E_PLUS"] - scores["E_MINUS"]),
            "O_margin": float(scores["O_PLUS"] - scores["O_MINUS"]),
        }
    }

# ============================================================
# Layer 3 (optional): NRC emotion profile (triangulation)
# ============================================================
def nrc_profile(text: str) -> dict:
    emo = NRCLex(text)
    total = sum(emo.raw_emotion_scores.values()) or 1
    return {k: float(v/total) for k, v in emo.raw_emotion_scores.items()}

# ============================================================
# Run analysis over all runs and keep an in-memory list for summaries
# ============================================================
records = []  # used by the next cell (pre-test tables)

for run in RUNS:
    in_dir = run["in_dir"]
    out_dir = run["out_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)

    if not in_dir.exists():
        print(f"[SKIP] {run['name']}: missing input dir {in_dir.resolve()}")
        continue

    files = sorted(in_dir.glob("*.json"))
    print(f"\n== Analysing run: {run['name']} ==")
    print(f"Input:  {in_dir.resolve()}")
    print(f"Output: {out_dir.resolve()}")
    print(f"Files:  {len(files)}")

    updated = 0
    skipped = 0

    for fp in files:
        raw = json.loads(fp.read_text(encoding="utf-8"))
        is_single = isinstance(raw, dict)
        recs = [raw] if is_single else raw

        for rec in recs:
            text = get_nested(rec, TEXT_PATH)
            if not isinstance(text, str) or not text.strip():
                skipped += 1
                continue

            variant = get_nested(rec, ("meta","variant"))
            product_id = get_nested(rec, ("meta","product_id"))
            model = get_nested(rec, ("generation","model"), default=None)

            # --- Empath bundles ---
            scores = empath_scores(text)
            for b, cats in BUNDLES.items():
                set_nested(rec, f"analysis.empath_bundles.{b}", bundle_mean(scores, cats))
                set_nested(rec, f"analysis.empath_bundle_meta.{b}.kept_categories", AUDIT[b]["kept"])
                set_nested(rec, f"analysis.empath_bundle_meta.{b}.dropped_categories", AUDIT[b]["dropped"])

            # --- Anchor similarity ---
            af = anchor_feats(text)
            set_nested(rec, "analysis.anchor_similarity.scores", af["scores"])
            set_nested(rec, "analysis.anchor_similarity.top_match", af["top_match"])
            set_nested(rec, "analysis.anchor_similarity.alignment_margin", af["alignment_margin"])
            set_nested(rec, "analysis.anchor_similarity.axis_margins", af["axis_margins"])
            if variant in ANCHORS:
                set_nested(rec, "analysis.anchor_similarity.correct_alignment", af["top_match"] == variant)
            else:
                set_nested(rec, "analysis.anchor_similarity.correct_alignment", None)

            # --- NRC emotions (optional) ---
            set_nested(rec, "analysis.nrc_emotions.profile", nrc_profile(text))

            # Store for summaries
            records.append({
                "run": run["name"],
                "file": fp.name,
                "rec": rec,
                "product_id": product_id,
                "variant": variant,
                "model": model,
                "text": text,
            })

        # write analysed copy (same filename) to out_dir
        out_path = out_dir / fp.name
        out_path.write_text(json.dumps(recs[0] if is_single else recs, ensure_ascii=False, indent=2), encoding="utf-8")
        updated += 1

    print(f"Done {run['name']}: wrote {updated} files. Skipped {skipped} records with missing text.")


Empath audit (dropped categories):
  E_social: OK (6 cats)
  E_energy: OK (5 cats)
  O_creative: OK (5 cats)
  O_abstract: OK (3 cats)
  O_practical: OK (7 cats)
  O_novelty: OK (4 cats)

== Analysing run: gpt_main ==
Input:  C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\Code\stimuli_out
Output: C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\Code\stimuli_out_analysed
Files:  78
Done gpt_main: wrote 78 files. Skipped 0 records with missing text.

== Analysing run: gpt_nano ==
Input:  C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\Code\stimuli_out\nano
Output: C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\Code\stimuli_out_analysed\nano
Files:  16
Done gpt_nano: wrote 16 files. Skipped 0 records with missing text.


In [ ]:
from pathlib import Path
from scipy.stats import binomtest
from statsmodels.stats.proportion import proportion_confint
import pandas as pd
import numpy as np

# ============================================================
# Pre-test reporting tables (per model run)
# Q1: Empath bundle sign tests across products
# Q2: Anchor alignment accuracy + confusion + axis-margin sign tests
# ============================================================

TARGET_VARIANTS = ["E_PLUS", "E_MINUS", "O_PLUS", "O_MINUS"]

# Build a tidy dataframe per record
rows = []
for r in records:
    rec = r["rec"]
    bundles = get_nested(rec, ("analysis","empath_bundles"), default={}) or {}
    margins = get_nested(rec, ("analysis","anchor_similarity","axis_margins"), default={}) or {}
    rows.append({
        "run": r["run"],
        "product_id": r["product_id"],
        "variant": r["variant"],
        "model": r["model"],
        "top_match": get_nested(rec, ("analysis","anchor_similarity","top_match")),
        "correct_alignment": get_nested(rec, ("analysis","anchor_similarity","correct_alignment")),
        "alignment_margin": get_nested(rec, ("analysis","anchor_similarity","alignment_margin")),
        **{k: bundles.get(k, np.nan) for k in BUNDLES.keys()},
        "E_margin": margins.get("E_margin", np.nan),
        "O_margin": margins.get("O_margin", np.nan),
    })

df = pd.DataFrame(rows)
print("Records loaded for summaries:", len(df))
display(df.groupby(["run","variant"]).size().rename("n_records").reset_index())

# ---------------- Utilities ----------------
def product_mean(tmp: pd.DataFrame, product_id: str, variant: str, col: str) -> float:
    vals = pd.to_numeric(tmp[(tmp.product_id==product_id) & (tmp.variant==variant)][col], errors="coerce").dropna()
    return float(vals.mean()) if len(vals) else np.nan

def sign_test_across_products(tmp: pd.DataFrame, a: str, b: str, col: str, expected: str) -> dict:
    # expected: "a_gt_b" or "a_lt_b"
    products = sorted(tmp.product_id.unique())
    flags = []
    for pid in products:
        ma = product_mean(tmp, pid, a, col)
        mb = product_mean(tmp, pid, b, col)
        if np.isnan(ma) or np.isnan(mb):
            continue
        flags.append(ma > mb if expected == "a_gt_b" else ma < mb)
    n = len(flags)
    k = int(sum(flags))
    p = binomtest(k=k, n=n, p=0.5, alternative="greater").pvalue if n else np.nan
    return {"contrast": f"{a} vs {b}", "bundle_or_axis": col, "successes": k, "n_products": n, "p_value": p}

def axis_sign_test(tmp: pd.DataFrame, variant: str, axis_col: str, expected_positive: bool) -> dict:
    products = sorted(tmp.product_id.unique())
    flags = []
    for pid in products:
        m = product_mean(tmp, pid, variant, axis_col)
        if np.isnan(m):
            continue
        flags.append(m > 0 if expected_positive else m < 0)
    n = len(flags)
    k = int(sum(flags))
    p = binomtest(k=k, n=n, p=0.5, alternative="greater").pvalue if n else np.nan
    return {"variant": variant, "axis": axis_col, "successes": k, "n_products": n, "p_value": p}

def accuracy_by_variant(tmp: pd.DataFrame) -> pd.DataFrame:
    out = []
    chance = 0.25
    for v in TARGET_VARIANTS + ["OVERALL"]:
        t = tmp[tmp.variant.isin(TARGET_VARIANTS)] if v == "OVERALL" else tmp[tmp.variant == v]
        t = t[t["correct_alignment"].notna()]
        n = len(t)
        k = int(t["correct_alignment"].sum()) if n else 0
        acc = k / n if n else np.nan
        ci_low, ci_high = (np.nan, np.nan)
        pval = np.nan
        if n:
            ci_low, ci_high = proportion_confint(k, n, alpha=0.05, method="wilson")
            pval = binomtest(k=k, n=n, p=chance, alternative="greater").pvalue
        out.append({
            "variant": v,
            "n": n,
            "k_correct": k,
            "accuracy": acc,
            "wilson_95_ci_low": ci_low,
            "wilson_95_ci_high": ci_high,
            "p_vs_chance_0.25": pval
        })
    return pd.DataFrame(out)

# ---------------- Produce tables per run ----------------
EXPORT_DIR = Path("pretest_exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for run_name, tmp in df.groupby("run"):
    print(f"\n==================== PRETEST: {run_name} ====================")

    # Table A: anchor alignment accuracy
    tabA = accuracy_by_variant(tmp)
    display(tabA)
    tabA.to_csv(EXPORT_DIR / f"{run_name}__tableA_anchor_accuracy.csv", index=False)

    # Table B: confusion matrix
    cm = pd.crosstab(
        tmp[tmp.variant.isin(TARGET_VARIANTS)]["variant"],
        tmp[tmp.variant.isin(TARGET_VARIANTS)]["top_match"],
        dropna=False
    )
    display(cm)
    cm.to_csv(EXPORT_DIR / f"{run_name}__tableB_confusion_matrix.csv")

    # Table C: axis margin sign tests (product-level)
    tabC = pd.DataFrame([
        axis_sign_test(tmp, "E_PLUS", "E_margin", expected_positive=True),
        axis_sign_test(tmp, "E_MINUS", "E_margin", expected_positive=False),
        axis_sign_test(tmp, "O_PLUS", "O_margin", expected_positive=True),
        axis_sign_test(tmp, "O_MINUS", "O_margin", expected_positive=False),
    ])
    display(tabC)
    tabC.to_csv(EXPORT_DIR / f"{run_name}__tableC_axis_margin_sign_tests.csv", index=False)

    # Table D: Empath bundle sign tests (product-level)
    tests = []
    tests += [sign_test_across_products(tmp, "E_PLUS", "E_MINUS", "E_social", expected="a_gt_b")]
    tests += [sign_test_across_products(tmp, "E_PLUS", "E_MINUS", "E_energy", expected="a_gt_b")]

    tests += [sign_test_across_products(tmp, "O_PLUS", "O_MINUS", "O_creative", expected="a_gt_b")]
    tests += [sign_test_across_products(tmp, "O_PLUS", "O_MINUS", "O_abstract", expected="a_gt_b")]
    tests += [sign_test_across_products(tmp, "O_PLUS", "O_MINUS", "O_novelty", expected="a_gt_b")]
    tests += [sign_test_across_products(tmp, "O_PLUS", "O_MINUS", "O_practical", expected="a_lt_b")]

    tabD = pd.DataFrame(tests).sort_values(["contrast","bundle_or_axis"])
    display(tabD)
    tabD.to_csv(EXPORT_DIR / f"{run_name}__tableD_empath_bundle_sign_tests.csv", index=False)

print(f"\nSaved pretest tables to: {EXPORT_DIR.resolve()}")


Records loaded for summaries: 94


,run,variant,n_records
0,gpt_main,E_MINUS,18
1,gpt_main,E_PLUS,18
2,gpt_main,GENERIC,6
3,gpt_main,O_MINUS,18
4,gpt_main,O_PLUS,18
5,gpt_nano,E_MINUS,4
6,gpt_nano,E_PLUS,3
7,gpt_nano,GENERIC,3
8,gpt_nano,O_MINUS,3
9,gpt_nano,O_PLUS,3



==================== PRETEST: gpt_main ====================


,variant,n,k_correct,accuracy,wilson_95_ci_low,wilson_95_ci_high,p_vs_chance_0.25
0,E_PLUS,18,15,0.833333,0.607780,0.941634,3.414461e-07
1,E_MINUS,18,16,0.888889,0.672002,0.968980,2.083834e-08
2,O_PLUS,18,10,0.555556,0.337164,0.754405,5.421779e-03
3,O_MINUS,18,5,0.277778,0.124998,0.508727,4.813307e-01
4,OVERALL,72,46,0.638889,0.523525,0.740183,3.845163e-12


top_match,E_MINUS,E_PLUS,O_MINUS,O_PLUS
variant,,,,
E_MINUS,16,1,0,1
E_PLUS,3,15,0,0
O_MINUS,9,4,5,0
O_PLUS,3,5,0,10


...

,variant,n,k_correct,accuracy,wilson_95_ci_low,wilson_95_ci_high,p_vs_chance_0.25
0,E_PLUS,3,3,1.0,0.438503,1.0,1.562500e-02
1,E_MINUS,4,4,1.0,0.510109,1.0,3.906250e-03
2,O_PLUS,3,3,1.0,0.438503,1.0,1.562500e-02
3,O_MINUS,3,3,1.0,0.438503,1.0,1.562500e-02
4,OVERALL,13,13,1.0,0.771905,1.0,1.490116e-08


top_match,E_MINUS,E_PLUS,O_MINUS,O_PLUS
variant,,,,
E_MINUS,4,0,0,0
E_PLUS,0,3,0,0
O_MINUS,0,0,3,0
O_PLUS,0,0,0,3


,variant,axis,successes,n_products,p_value
0,E_PLUS,E_margin,1,1,0.5
1,E_MINUS,E_margin,1,1,0.5
2,O_PLUS,O_margin,1,1,0.5
3,O_MINUS,O_margin,1,1,0.5


,contrast,bundle_or_axis,successes,n_products,p_value
1,E_PLUS vs E_MINUS,E_energy,1,1,0.5
0,E_PLUS vs E_MINUS,E_social,1,1,0.5
3,O_PLUS vs O_MINUS,O_abstract,1,1,0.5
2,O_PLUS vs O_MINUS,O_creative,1,1,0.5
4,O_PLUS vs O_MINUS,O_novelty,1,1,0.5
5,O_PLUS vs O_MINUS,O_practical,1,1,0.5



Saved pretest tables to: C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\Code\pretest_exports




| variant | n | k_correct | accuracy | wilson_95_ci_low | wilson_95_ci_high | p_vs_chance_0.25 |
|---------|---|-----------|----------|------------------|-------------------|-------------------|
| E_PLUS | 18 | 15 | 0.8333333333333334 | 0.6077796189608428 | 0.9416342319413856 | 3.414461389183998e-07 |
| E_MINUS | 18 | 16 | 0.8888888888888888 | 0.672002348698212 | 0.9689804773543876 | 2.0838342607021332e-08 |
| O_PLUS | 18 | 10 | 0.5555555555555556 | 0.33716415642042774 | 0.7544048187299437 | 0.0054217792057897896 |
| O_MINUS | 18 | 5 | 0.2777777777777778 | 0.12499753379553993 | 0.5087265656029747 | 0.4813306695432402 |
| OVERALL | 72 | 46 | 0.6388888888888888 | 0.5235248046738358 | 0.7401832029382592 | 3.845162674011787e-12 |



| variant | E_MINUS | E_PLUS | O_MINUS | O_PLUS |
|---------|---------|--------|---------|--------|
| E_MINUS | 16 | 1 | 0 | 1 |
| E_PLUS | 3 | 15 | 0 | 0 |
| O_MINUS | 9 | 4 | 5 | 0 |
| O_PLUS | 3 | 5 | 0 | 10 |



| variant | axis | successes | n_products | p_value |
|---------|------|-----------|------------|---------|
| E_PLUS | E_margin | 5 | 6 | 0.109375 |
| E_MINUS | E_margin | 6 | 6 | 0.015625 |
| O_PLUS | O_margin | 6 | 6 | 0.015625 |
| O_MINUS | O_margin | 5 | 6 | 0.109375 |



| contrast | bundle_or_axis | successes | n_products | p_value |
|----------|----------------|-----------|------------|---------|
| E_PLUS vs E_MINUS | E_energy | 5 | 6 | 0.109375 |
| E_PLUS vs E_MINUS | E_social | 6 | 6 | 0.015625 |
| O_PLUS vs O_MINUS | O_abstract | 5 | 6 | 0.109375 |
| O_PLUS vs O_MINUS | O_creative | 6 | 6 | 0.015625 |
| O_PLUS vs O_MINUS | O_novelty | 4 | 6 | 0.34375 |
| O_PLUS vs O_MINUS | O_practical | 4 | 6 | 0.34375 |


The core goal of the stimulus pre-test was **manipulation validity**: when we generate four message variants (E_PLUS, E_MINUS, O_PLUS, O_MINUS), do the texts *actually* contain the intended psychological and linguistic signals, both at the level of **word choice** (lexical cues) and at the level of **overall meaning/tone** (semantic cues)?

This matters because in personality-targeted persuasion, effects depend on whether messages reliably express the intended “dial settings” rather than merely *sounding different* in arbitrary ways. This is also why recent work emphasizes careful prompt and output validation: small prompt changes can shift what the model produces, so you want evaluation that is robust to superficial artifacts [10].

To do that, we used two complementary measurement families:
1. **Lexicon-based psycholinguistic signals** (Empath bundles; “what words appear?”)
2. **Embedding-based semantic signals** (anchor similarity; “what does the whole text feel like?”) 

That combination mirrors the broader idea in schema-based / psycholinguistic steering work: you want interpretable, measurable dimensions—and you want to verify them with multiple lenses rather than relying on a single metric [5].

---
## 1) Empath bundle scores (lexicon-based “feature presence”)

  ### What it is (briefly)

Empath is a lexicon method: it counts words associated with psychologically meaningful categories (e.g., *social*, *helping*, *energy*). We grouped categories into **bundles** aligned with the intended traits:
* **E_social**, **E_energy** for Extraversion-related cues
* **O_creative**, **O_abstract**, **O_practical**, **O_novelty** for Openness-related cues

### Why we chose it

Empath-style measures are useful because they are: 
* **interpretable** (you can say *which* psychological facets are present), and
* **diagnostic** (they show *what kind* of language shifted, not just that something shifted).

This is especially valuable in persuasion/personalization contexts, where prior work shows that language cues and framing often carry the effect (not just message length or sentiment) [1].

### How to interpret the values

You’ll typically see the Empath analysis summarized not just as raw scores, but as **directional tests** (“does the intended condition have more of the intended bundle than its opposite?”). That is what Table D reports.

---
## 2) Anchor similarity & correctness (embedding-based “whole-message tone”)

### What it is (briefly)

We represent each message using a sentence embedding model (a transformer that maps text into a vector space). We then compare each message to **anchor texts** that represent the four targets (E_PLUS, E_MINUS, O_PLUS, O_MINUS). The predicted label is whichever anchor is closest.

This gives two outputs:
* **Accuracy**: how often the closest anchor matches the intended label
* **Similarity margins**: how much closer the intended anchor is than the opposite pole on the same axis
### Why we chose it

Lexicon counts can miss *global meaning*: a message can avoid obvious keywords but still sound energetic or analytical. Embeddings capture higher-level semantics, tone, framing, and topic coherence, more directly than word counts.


This is important because LLMs can express a persona in ways that are not tied to a small, predictable vocabulary. Using a semantic check helps verify that the manipulation holds at the level that readers (or downstream models) would perceive [5].

### How to interpret the values

* **Chance accuracy** with 4 classes is **0.25**
* “Good” outcomes look like **accuracy clearly above 0.25**, ideally **70–95%**, with statistical evidence that it beats chance.

---
## 3) Confusion matrix (what the mistakes look like)

### What it is

A confusion matrix shows, for each intended label (row), what the classifier predicted (columns). It tells you whether errors are random or systematic.

### Why we chose it

Overall accuracy can hide *structured failure modes*. For example, if O_MINUS is consistently mistaken for E_MINUS, that’s not “noise”—it reveals that the generation prompt or the feature space is blending those concepts.

---
## 4) Margin sign tests (nonparametric directional reliability)

### What it is

Instead of relying on the magnitude of similarity differences, we ask a simpler question per product:

> For this product, is the text closer to the intended pole than to its opposite pole?

We then count how often this is true (“successes”) across products and run a **sign test** (binomial, nonparametric).

### Why we chose it

Similarity scores can be non-normal and can vary by product/topic. The sign test is robust: it only needs the *direction* of the effect, not distributional assumptions.

# Now: interpreting CSV results

Below I interpret the concrete values from our four output tables.

| variant | n   | k_correct | accuracy           | wilson_95_ci_low    | wilson_95_ci_high  | p_vs_chance_0.25       |
| ------- | --- | --------- | ------------------ | ------------------- | ------------------ | ---------------------- |
| E_PLUS  | 18  | 15        | 0.8333333333333334 | 0.6077796189608428  | 0.9416342319413856 | 3.414461389183998e-07  |
| E_MINUS | 18  | 16        | 0.8888888888888888 | 0.672002348698212   | 0.9689804773543876 | 2.0838342607021332e-08 |
| O_PLUS  | 18  | 10        | 0.5555555555555556 | 0.33716415642042774 | 0.7544048187299437 | 0.0054217792057897896  |
| O_MINUS | 18  | 5         | 0.2777777777777778 | 0.12499753379553993 | 0.5087265656029747 | 0.4813306695432402     |
| OVERALL | 72  | 46        | 0.6388888888888888 | 0.5235248046738358  | 0.7401832029382592 | 3.845162674011787e-12  |
## Table A — Anchor accuracy (semantic classification)

**Key results (n=18 per variant; total n=72):**

* **E_PLUS:** 15/18 correct → **accuracy = 0.833**, p vs chance 0.25 = **3.41e-07**
* **E_MINUS:** 16/18 correct → **accuracy = 0.889**, p = **2.08e-08**
* **O_PLUS:** 10/18 correct → **accuracy = 0.556**, p = **0.00542**
* **O_MINUS:** 5/18 correct → **accuracy = 0.278**, p = **0.481**
* **Overall:** 46/72 correct → **accuracy = 0.639**, p = **3.85e-12**
### Interpretation

* The **Extraversion axis is strongly and cleanly realized**: both E_PLUS and E_MINUS are far above chance and highly significant.

* **O_PLUS is detectable** above chance (moderate accuracy, significant).

* **O_MINUS fails the semantic test**: 0.278 is essentially chance, and the p-value confirms it is *not reliably distinguishable* from random guessing.

**What “good” looks like here:**

Your E variants are in the “good” range (strong, stable). O_PLUS is “okay but weaker.” O_MINUS is “not good” (the semantic signature is not consistently present).

---
## Table B — Confusion matrix (where O_MINUS goes wrong)
Rows are the true labels; columns are predicted labels:

| variant | E_MINUS | E_PLUS | O_MINUS | O_PLUS |
| ------- | ------- | ------ | ------- | ------ |
| E_MINUS | 16      | 1      | 0       | 1      |
| E_PLUS  | 3       | 15     | 0       | 0      |
| O_MINUS | 9       | 4      | 5       | 0      |
| O_PLUS  | 3       | 5      | 0       | 10     |
* **E_MINUS:** predicted correctly 16 times; only 2 total errors (1→E_PLUS, 1→O_PLUS)
* **E_PLUS:** predicted correctly 15 times; main error is 3→E_MINUS
* **O_PLUS:** predicted correctly 10 times; most confusion is **5→E_PLUS** (and 3→E_MINUS)
* **O_MINUS:** predicted correctly only 5 times; most confusion is **9→E_MINUS** and *4→E_PLUS**
### Interpretation
The errors are *not random*:

* **O_MINUS is systematically read as Extraversion-negative / low-energy language** (mostly E_MINUS).

* This strongly suggests that our O_MINUS generation is producing language that embedding space treats as “introverted/low-energy” rather than “low-openness.”

In other words: the pipeline successfully separates **E+ vs E−**, but it tends to map **O− into the E− region**, collapsing two intended constructs.  

---
## Table C — Axis margin sign tests (robust directional check)

Each variant is evaluated across **6 products** (“n_products = 6”). “successes” counts how often the intended pole is closer than its opposite.

   **E_PLUS:** 5/6 successes, p = **0.109**
* **E_MINUS:** 6/6 successes, p = **0.0156**
* **O_PLUS:** 6/6 successes, p = **0.0156**
* **O_MINUS:** 5/6 successes, p = **0.109**

### Interpretation
  
This gives a more nuanced view than accuracy:  

* **E_MINUS and O_PLUS are directionally consistent across all products** (6/6) and pass p < 0.05.
* **E_PLUS and O_MINUS are *mostly* consistent (5/6)** but don’t reach significance with only 6 product-level tests (p=0.109 is suggestive but not strong evidence).

This pattern is common in small-n sign tests: with n=6, you typically need **6/6** to cross 0.05.

---
## Table D — Empath bundle sign tests (facet-level construct checks)
Again, 6 products. These tests compare *within an axis*:
### Extraversion bundles (E_PLUS vs E_MINUS)

* **E_social:** 6/6 successes, p = **0.0156**
* **E_energy:** 5/6 successes, p = **0.109** (trend, not significant)
### Openness bundles (O_PLUS vs O_MINUS)  

* **O_creative:** 6/6 successes, p = **0.0156**
* **O_abstract:** 5/6 successes, p = **0.109** (trend)
* **O_practical:** 4/6 successes, p = **0.344**
* **O_novelty:** 4/6 successes, p = **0.344**
### Interpretation
Facet-by-facet, you get **clear evidence** for:  

* **E_social** being reliably higher in E_PLUS than E_MINUS
* **O_creative** being reliably higher in O_PLUS than O_MINUS  

But the other bundles are weaker or inconsistent:
* E_energy and O_abstract show directional trends but do not meet p < 0.05 with n=6

* O_practical and O_novelty do not show reliable separation (4/6 is close to chance directionally)

This suggests our manipulation reliably drives **some** theoretically aligned facets (sociality, creativity), but does not consistently move **all** facets of the broader trait bundle—especially on the Openness side.

---
# So… are these values “good,” and what did we hope to achieve?

### What we hoped to achieve

Ideally, we wanted converging evidence that:
1. **Semantic separation works** (high anchor accuracy; strong diagonal confusion matrix)
2. **Directional separation is robust across products** (margin sign tests significant)
3. **Lexical/psychological facets shift in the intended directions** (Empath bundle tests significant) 
### What we actually achieved (summary)

**Strong success on Extraversion (E axis):**
* High semantic accuracy for both poles (0.83–0.89; extremely low p-values)
* Confusion matrix is close to diagonal for E labels
* Robust margin direction especially for E_MINUS (6/6)
* Clear lexical facet evidence for E_social (6/6)  

**Partial success on Openness (O axis), but asymmetric:**
* O_PLUS is detectable (0.56 accuracy; significant)
* O_MINUS is not semantically reliable (0.28; not significant), and is systematically confused with E_MINUS
* Lexically, O_creative separates reliably (6/6), but novelty/practicality do not
### Are these “good” values?  

* **Yes, for E_PLUS/E_MINUS**: these are the kind of validation numbers you can confidently report as a successful manipulation.
* **Mixed for O_PLUS**: it works, but weaker than Extraversion.
* **No for O_MINUS (as currently generated)**: the pipeline does not produce a stable “low-openness” semantic signature; instead it collapses toward low-energy/introversion-like language.
---
# What we can prove / show from these values

Given these results, the strongest defensible claims are:

1. **We can reliably steer Extraversion language in both directions.**
   Both semantic and lexical diagnostics converge: messages intended as E_PLUS and E_MINUS are distinguishable far above chance, with minimal confusion.  

2. **We can steer “high Openness” (O_PLUS) moderately well, including creativity-related cues.**
   O_PLUS is above chance semantically and shows strong lexical evidence on O_creative.

3. **Our “low Openness” (O_MINUS) manipulation is not validated and likely confounded with low-energy language.**

   The confusion matrix shows systematic misclassification into E_MINUS, and semantic accuracy is not above chance. This is important: it doesn’t mean the model *can’t* express O_MINUS, but it means **our current prompt/realization did not reliably instantiate it**.  

4. **The manipulation is facet-specific rather than uniformly trait-wide.**

   Even when a broad trait works (especially Openness), only certain sub-facets (e.g., creativity, abstraction trend) show consistent separation, while others (novelty, practicality) do not. This is informative because it identifies which components of the construct are actually being expressed in the generated language.

> “Overall, the validation analyses provide strong convergent evidence that our generation procedure successfully instantiated the intended Extraversion manipulation (both lexically and semantically), while Openness showed an asymmetric pattern: O_PLUS was moderately detectable—especially via creativity-related cues—whereas O_MINUS was not semantically separable from chance and was systematically confused with low-energy (E_MINUS) language.”


---
[[References]]

[1]  Matz, S. C., Teeny, J. D., Vaid, S. S., Peters, H., Harari, G. M., & Cerf, M. (2024). The potential of generative AI for personalized persuasion at scale. *Scientific Reports, 14*, 4692. [https://doi.org/10.1038/s41598-024-53755-0](https://doi.org/10.1038/s41598-024-53755-0)  
[2] Mieleszczenko-Kowszewicz, W., Płudowski, D., Kołodziejczyk, F., Świstak, J., Sienkiewicz, J., & Biecek, P. (2024). The dark patterns of personalized persuasion in large language models: Exposing persuasive linguistic features for Big Five personality traits in LLMs responses. _Proceedings of the ACM Web Conference_. [https://doi.org/XXXXXXXXXXXXXX](https://doi.org/XXXXXXXXXXXXXX) _(Note: The DOI was incomplete in the provided paper)_  
[3] Ta, V. P., Boyd, R. L., Seraj, S., Keller, A., Griffith, C., Loggarakis, A., & Medema, L. (2022). An inclusive, real-world investigation of persuasion in language and verbal behavior. _Journal of Computational Social Science, 5_(1), 883–903. [https://doi.org/10.1007/s42001-021-00153-5](https://doi.org/10.1007/s42001-021-00153-5)  
[4] Yeo, G. C., Churina, S., & Jaidka, K. (2025). Do you trust me? Cognitive-affective signatures of trustworthiness in large language models. _Proceedings of the ACM Web Conference_. [https://doi.org/XXXXXXX.XXXXXXX](https://doi.org/XXXXXXX.XXXXXXX) _(Note: The DOI was incomplete in the provided paper)_  
[5] Cisar, C., Sheffield, E., Drake, J., Harrell, A., Chidambaram, S., Nangia, N., Arannil, V., & Williams, A. (2025). *PILOT: Steering synthetic data generation with psychological & linguistic output targeting* (arXiv:2509.15447). arXiv.  
[6] Argyle, L. P., Busby, E. C., Fulda, N., Gubler, J., Rytting, C., & Wingate, D. (2022). *Out of one, many: Using language models to simulate human samples* (arXiv:2209.06899). arXiv.  
[7] Bai, H., Voelkel, J. G., Muldowney, S., Eichstaedt, J. C., & Willer, R. (2025). LLM-generated messages can persuade humans on policy issues. *Nature Communications, 16*, 6037. [https://doi.org/10.1038/s41467-025-61345-5](https://doi.org/10.1038/s41467-025-61345-5)  
[8] Bisbee, J., Clinton, J. D., Dorff, C., Kenkel, B., & Larson, J. M. (2024). Synthetic replacements for human survey data? The perils of large language models. *Political Analysis*. [https://doi.org/10.1017/pan.2024.5](https://doi.org/10.1017/pan.2024.5)  
[9] Chaves, A. P., Egbert, J., Hocking, T., Doerry, E., & Gerosa, M. A. (2021). *Chatbots language design: The influence of language variation on user experience* (arXiv:2101.11089). arXiv.  
[10] Lutz, M., Sen, I., Ahnert, G., Rogers, E., & Strohmaier, M. (2025). *The prompt makes the person(a): A systematic evaluation of sociodemographic persona prompting for large language models* (arXiv:2507.16076). arXiv.  
[11] Meguellati, E., Civelli, S., Han, L., Bernstein, A., Sadiq, S., & Demartini, G. (2025). *LLM-generated ads: From personalization parity to persuasion superiority* (arXiv:2512.03373). arXiv.  
[12] Saini, H., Tang, Y., & Liu, D. (2026). *Bridging mechanistic interpretability and prompt engineering with gradient ascent for interpretable persona control* (arXiv:2601.02896). arXiv.  
[13] Weeber, F., Neplenbroek, V., Batzner, J., & Padó, S. (2026). *One persona, many cues, different results: How sociodemographic cues impact LLM personalization* (arXiv:2601.18572). arXiv.